# What a prompt actually changes

MichAl Academy, unit 5.1.

Run each cell with **Shift+Enter**.

Prompting advice is mostly folklore, repeated because somebody tried it once
and the answer got better. This notebook does the boring thing instead: the
same task, twenty-four items, one variable changed at a time, scored.

The model is **SmolLM2-135M-Instruct**, which is small and therefore honest. A
large model is good enough to survive a badly built prompt, so it hides the
effect being measured. Everything here gets larger as the model gets smaller,
and the direction of every result is the part that transfers.


In [ ]:
import warnings

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME)
model.eval()


@torch.no_grad()
def ask(messages, n=8):
    """One turn of conversation, answered greedily so nothing here is luck."""
    ids = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt")
    out = model.generate(ids, max_new_tokens=n, do_sample=False,
                         pad_token_id=tok.eos_token_id,
                         attention_mask=torch.ones_like(ids))
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


print(ask([{"role": "user", "content": "Say the word ready and nothing else."}]))


## The task

Twenty-four support messages, each clearly one of three kinds. The task is easy
on purpose: if the model cannot do it at all, nothing about the prompt can be
measured, because every result would be zero.

Two things get scored separately, and keeping them apart is the point of the
whole unit:

- **format** is whether the reply is exactly one of the three labels
- **correct** is whether the right label is in the reply anywhere

A reply of "Billing: this message is about a payment" is **correct** and badly
**formatted**. Software downstream cares about the second one.


In [ ]:
LABELS = ["billing", "delivery", "technical"]

ITEMS = [
    ("My card was charged twice for the same order.", "billing"),
    ("The parcel says delivered but nothing arrived.", "delivery"),
    ("The app crashes every time I open the settings.", "technical"),
    ("I was invoiced for a subscription I cancelled in March.", "billing"),
    ("Tracking has not updated in six days.", "delivery"),
    ("I cannot log in, it says my password is wrong after a reset.", "technical"),
    ("Why is there a 4 pound fee on my statement?", "billing"),
    ("The courier left the box at the wrong house number.", "delivery"),
    ("Video playback stops after ten seconds on wifi.", "technical"),
    ("Please refund the difference after the price drop.", "billing"),
    ("My order shipped to my old address.", "delivery"),
    ("The export button does nothing in Firefox.", "technical"),
    ("You took the annual fee again after I downgraded.", "billing"),
    ("The driver marked it undeliverable without knocking.", "delivery"),
    ("Two-factor codes never arrive on my phone.", "technical"),
    ("I was promised a credit note and it never appeared.", "billing"),
    ("It has been sitting at the sorting centre since Tuesday.", "delivery"),
    ("The search results page loads forever and then times out.", "technical"),
    ("The discount code was not applied at checkout.", "billing"),
    ("Half the order arrived and the rest is missing.", "delivery"),
    ("Uploading a photo fails with an unknown error.", "technical"),
    ("My statement shows a currency conversion charge nobody mentioned.", "billing"),
    ("The estimated date has moved back three times.", "delivery"),
    ("The mobile app logs me out every few minutes.", "technical"),
]

SHOTS = [
    ("The payment failed but the money left my account.", "billing"),
    ("It has been stuck at the depot for a week.", "delivery"),
    ("The page is blank after I sign in.", "technical"),
]

print(f"{len(ITEMS)} items, {len(LABELS)} labels, chance is 1 in {len(LABELS)}")


def score(build):
    exact = correct = 0
    samples = []
    for text, gold in ITEMS:
        reply = ask(build(text))
        clean = reply.strip().strip(".").strip().lower()
        if clean in LABELS:
            exact += 1
            if clean == gold:
                correct += 1
        elif gold in reply.lower():
            correct += 1
        if len(samples) < 2:
            samples.append(reply)
    return exact, correct, samples


## Four prompts

The same instruction, in four shapes.

1. **Instruction only.** Say what to do.
2. **Instruction plus a format rule.** Say what to do and how to answer.
3. **Three examples pasted into the message.** The shape most prompting guides
   show, written as `Message: ... Label: ...` lines.
4. **The same three examples as conversation turns.** Each example is a real
   user turn with a real assistant turn answering it.

Three and four contain identical examples and identical words. The only thing
that differs is where they sit.


In [ ]:
RULE = ("Classify the support message as one of: billing, delivery, technical.\n"
        "Answer with the single label and nothing else.")


def plain(text):
    return [{"role": "user", "content":
             "Classify this support message as billing, delivery or technical."
             f"\n\nMessage: {text}"}]


def with_rule(text):
    return [{"role": "user", "content": f"{RULE}\n\nMessage: {text}"}]


def shots_inline(text):
    block = "".join(f"Message: {m}\nLabel: {lab}\n\n" for m, lab in SHOTS)
    return [{"role": "user", "content": f"{RULE}\n\n{block}Message: {text}\nLabel:"}]


def shots_as_turns(text):
    msgs = [{"role": "user", "content": f"{RULE}\n\nMessage: {SHOTS[0][0]}"},
            {"role": "assistant", "content": SHOTS[0][1]}]
    for m, lab in SHOTS[1:]:
        msgs.append({"role": "user", "content": f"Message: {m}"})
        msgs.append({"role": "assistant", "content": lab})
    msgs.append({"role": "user", "content": f"Message: {text}"})
    return msgs


n = len(ITEMS)
print(f"{'':<34}{'format':>8}{'correct':>9}")
for name, build in (
    ("instruction only", plain),
    ("instruction + format rule", with_rule),
    ("3 examples, pasted in the message", shots_inline),
    ("3 examples, as conversation turns", shots_as_turns),
):
    exact, correct, samples = score(build)
    print(f"{name:<34}{exact:>6}/{n}{correct:>7}/{n}")
    for s in samples:
        print(f"{'':<36}{s[:58]!r}")


## What that says

**Pasting examples into the message destroyed the task.** Zero right, zero
correctly formatted, and the replies are the model continuing the pattern by
repeating an example back rather than answering. The words were right and the
placement was wrong.

**The same examples as conversation turns fixed the formatting.** A chat model
was trained on alternating user and assistant turns, so an example written as
one is an example in the shape it recognises.

**And they did not make it better at the task.** Compare the correct column for
"instruction only" against "as conversation turns". The examples bought
formatting, which is real and worth having, and they did not buy accuracy.

That distinction is the useful part of this unit. Few-shot examples teach the
**shape** of an answer. If the model does not know the subject, showing it three
answers does not teach it the subject.


## What the words cost

One more measurement, because prompt length is the thing everyone forgets is
priced. Unit 4.1.3 charged everything in tokens; a prompt is tokens too, and it
is paid on every single request.


In [ ]:
one = ITEMS[0][0]
for name, build in (
    ("instruction only", plain),
    ("instruction + format rule", with_rule),
    ("3 examples, pasted", shots_inline),
    ("3 examples, as turns", shots_as_turns),
):
    ids = tok.apply_chat_template(build(one), add_generation_prompt=True,
                                  return_tensors="pt")
    print(f"{name:<28}{ids.shape[1]:>5} tokens")


## Advice that does not survive being measured

Six versions of the same prompt. One plain, five carrying a piece of advice you
will have seen repeated: be polite, say it is urgent, threaten a penalty, assign
an expert role, ask it to take a deep breath.

Watch the spread rather than the winner.


In [ ]:
PREFIX = {
    "plain": "",
    "polite": "Please, if you would be so kind. ",
    "urgent": "This is extremely important and urgent. ",
    "threat": "You will be penalised if you get this wrong. ",
    "expert role": "You are an expert support analyst. ",
    "deep breath": "Take a deep breath and work carefully. ",
}

for name, pre in PREFIX.items():
    got = 0
    for text, gold in ITEMS:
        prompt = f"{pre}{RULE}\n\nMessage: {text}"
        reply = ask([{"role": "user", "content": prompt}], n=10)
        clean = reply.strip().strip(".").strip().lower()
        got += (clean == gold) if clean in LABELS else (gold in reply.lower())
    print(f"{name:<14}{got:>3}/{len(ITEMS)}")


Every one lands within a couple of items of every other. On twenty-four items at
this success rate, one standard error is about two items, so the whole spread is
inside the noise.

**And that is how prompting folklore is made.** Whichever line came out two
ahead of plain, run once, reads as a trick that works. Run the plain prompt a
second time on different items and it changes places.


## Splitting a task into steps

Chain of thought in unit 4.6.1 bought real accuracy on a question that needed
two hops. This task needs one. Here the model is asked what the message is
about, and that answer is fed into the labelling prompt.


In [ ]:
def direct(t):
    return [{"role": "user", "content": f"{RULE}\n\nMessage: {t}"}]


def two_step(t):
    summary_prompt = f"Message: {t}\n\nIn three words, what is this message about?"
    about = ask([{"role": "user", "content": summary_prompt}], n=12)
    prompt = f"{RULE}\n\nMessage: {t}\nWhat it is about: {about}"
    return [{"role": "user", "content": prompt}]


for name, build in (("one step", direct), ("two steps", two_step)):
    got = 0
    for text, gold in ITEMS:
        reply = ask(build(text), n=10)
        clean = reply.strip().strip(".").strip().lower()
        got += (clean == gold) if clean in LABELS else (gold in reply.lower())
    print(f"{name:<12}{got:>3}/{len(ITEMS)}")


Splitting it made things considerably worse, and the reason is visible in what
the first step produces. Its three-word summary is sometimes wrong, and a wrong
summary sitting in the prompt is a confident wrong answer the second step now
has to argue with.

**A step you add is a step that can fail.** Decomposition pays when the task
genuinely has stages that each need the previous one's answer. On a task that
was already one step, it adds a failure point and buys nothing.


## What this unit measured

- **Where an example sits matters more than what it says.** The same three
  examples were worth nothing pasted into a message and worth most of the
  formatting as conversation turns.
- **Format compliance and accuracy are separate.** A prompt change can buy a
  great deal of one and none of the other, and reporting a single "it got
  better" hides which.
- **Every prompt is paid for on every request**, and one cell prices the four
  in tokens.
- **None of the folk remedies moved the score** beyond the run-to-run noise,
  and the one that came out two ahead is exactly how such advice gets believed.
- **Splitting a one-step task into two made it worse**, because a wrong
  intermediate answer becomes a confident premise for the next step.
